# PCL Detection — RoBERTa-base Fine-tuning
### NLP Coursework, Exercise 4 — Ivan Artiukhov

**Approach:** RoBERTa-base fine-tuned with inverse-frequency class-weighted cross-entropy loss and dev-set decision threshold tuning. Targets F1 > 0.48 (RoBERTa-base baseline).

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save

Steps:
1. Install dependencies
2. Upload data files (`dontpatronizeme_pcl.tsv`, `train_semeval_parids-labels.csv`, `dev_semeval_parids-labels.csv`, `task4_test.tsv`)
3. Configure and load data
4. Tokenise and build datasets
5. Set up class weights, model, and optimiser
6. Train — saves best checkpoint by dev F1
7. Load best checkpoint
8. Tune decision threshold on dev set
9. Write `dev.txt`
10. Generate and write `test.txt`
11. Download output files

In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers torch scikit-learn pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ── 2. Mount Google Drive and set data path ──────────────────────────────────
# Before running this cell:
#   1. Go to https://drive.google.com
#   2. Create a folder called "NLP_CW" (My Drive → New → Folder)
#   3. Upload all 4 data files into that folder:
#        dontpatronizeme_pcl.tsv
#        train_semeval_parids-labels.csv
#        dev_semeval_parids-labels.csv
#        task4_test.tsv
#   4. Then run this cell — click "Connect to Google Drive" when prompted.
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/NLP_CW'
print(f'Data directory: {DATA_DIR}')
print('Files found:')
import os
for f in ['dontpatronizeme_pcl.tsv', 'train_semeval_parids-labels.csv',
          'dev_semeval_parids-labels.csv', 'task4_test.tsv']:
    path = os.path.join(DATA_DIR, f)
    status = '✓' if os.path.exists(path) else '✗  MISSING'
    print(f'  {status}  {f}')

ModuleNotFoundError: No module named 'google'

In [ ]:
# ── 3. Imports and config ────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME   = 'roberta-base'
MAX_LENGTH   = 128
BATCH_SIZE   = 16
EPOCHS       = 10
LR           = 2e-5
WARMUP_FRAC  = 0.06   # shorter warmup so full LR kicks in faster
BEST_CKPT    = 'best_model.pt'

# Device: CUDA (Colab GPU) → MPS (Apple Silicon) → CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: No GPU detected. Training will be very slow. Enable T4 GPU in Runtime settings.')
elif DEVICE.type == 'mps':
    print('Running on Apple Silicon MPS. Consider using Colab T4 GPU for faster training.')

In [ ]:
# ── 4. Load and prepare data ─────────────────────────────────────────────────
def load_tsv(path):
    records = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t', 5)
            if len(parts) == 6 and parts[0].strip().isdigit():
                pid  = int(parts[0])
                text = parts[4].strip()
                lbl  = int(parts[5])
                records[pid] = (text, lbl)
    return records

def load_test_tsv(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t', 5)
            if len(parts) >= 5 and parts[0].strip().startswith('t_'):
                records.append((parts[0].strip(), parts[4].strip()))
    return records

def binarise(label_raw):
    return 1 if label_raw >= 2 else 0

def build_split(par_ids, records):
    texts, labels = [], []
    for pid in par_ids:
        text, lbl_raw = records[pid]
        texts.append(text)
        labels.append(binarise(lbl_raw))
    return texts, labels

print('Loading data...')
records   = load_tsv(os.path.join(DATA_DIR, 'dontpatronizeme_pcl.tsv'))
train_ids = list(pd.read_csv(os.path.join(DATA_DIR, 'train_semeval_parids-labels.csv'))['par_id'])
dev_ids   = list(pd.read_csv(os.path.join(DATA_DIR, 'dev_semeval_parids-labels.csv'))['par_id'])

train_texts, train_labels = build_split(train_ids, records)
dev_texts,   dev_labels   = build_split(dev_ids,   records)

print(f'  Train: {len(train_texts)} samples  |  PCL={sum(train_labels)} ({sum(train_labels)/len(train_labels)*100:.1f}%)')
print(f'  Dev  : {len(dev_texts)} samples   |  PCL={sum(dev_labels)} ({sum(dev_labels)/len(dev_labels)*100:.1f}%)')

In [ ]:
# ── 5. Tokenise and build datasets ───────────────────────────────────────────
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

class PCLDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        self.labels = labels

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = PCLDataset(train_texts, train_labels)
dev_dataset   = PCLDataset(dev_texts,   dev_labels)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader    = DataLoader(dev_dataset,   batch_size=BATCH_SIZE)

print('Datasets ready.')

In [4]:
# (duplicate — skip this cell)

Device: cpu


In [ ]:
# (duplicate — skip this cell)

In [ ]:
# (duplicate — skip this cell)

In [ ]:
# ── 6. Class weights, model, optimiser ──────────────────────────────────────
n_total = len(train_labels)
n_pcl   = sum(train_labels)
n_npcl  = n_total - n_pcl

# Asymmetric weighting: keep No-PCL at 1.0, upweight PCL by the class ratio.
# The balanced formula (N/2N_c) creates near-zero net gradients; this avoids that.
w0 = 1.0
w1 = n_npcl / n_pcl   # raw imbalance ratio ≈ 9.5
class_weights = torch.tensor([w0, w1], dtype=torch.float).to(DEVICE)
print(f'Class ratio: {n_npcl}:{n_pcl} ({n_npcl/n_pcl:.1f}x)')
print(f'Class weights  →  No-PCL: {w0:.4f}  |  PCL: {w1:.4f}')

model     = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

loss_fn      = nn.CrossEntropyLoss(weight=class_weights)
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print('Model loaded.')

In [ ]:
# ── 7. Training loop ─────────────────────────────────────────────────────────
def get_probs(loader):
    """Return (probs, labels) arrays for the full loader."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop('labels').to(DEVICE)
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            probs  = torch.softmax(model(**batch).logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_probs), np.array(all_labels)

def best_tuned_f1(probs, labels):
    """Search thresholds in [0.1, 0.9] and return (best_f1, best_threshold)."""
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.10, 0.91, 0.01):
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_f1, best_t

print('Training...\n')
best_tuned = 0.0   # tracks best tuned F1 across epochs for checkpoint

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        labels = batch.pop('labels').to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        loss   = loss_fn(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    dev_probs, dev_true = get_probs(dev_loader)
    f1_at_half = f1_score(dev_true, (dev_probs >= 0.5).astype(int), pos_label=1, zero_division=0)
    tuned_f1, tuned_t = best_tuned_f1(dev_probs, dev_true)
    mean_pos = dev_probs[dev_true == 1].mean()
    mean_neg = dev_probs[dev_true == 0].mean()

    saved = ''
    if tuned_f1 >= best_tuned:
        best_tuned = tuned_f1
        torch.save(model.state_dict(), BEST_CKPT)
        saved = '  ← saved'

    print(f'Ep {epoch:02d}/{EPOCHS}  loss={avg_loss:.4f}  '
          f'F1@0.5={f1_at_half:.4f}  tuned_F1={tuned_f1:.4f}(t={tuned_t:.2f})  '
          f'P(PCL): pos={mean_pos:.3f} neg={mean_neg:.3f}{saved}')

print(f'\nBest tuned dev F1: {best_tuned:.4f}')

In [ ]:
# ── 8. Load best checkpoint ──────────────────────────────────────────────────
print(f'Loading best checkpoint (best tuned dev F1={best_tuned:.4f})...')
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
print('Done.')

In [ ]:
# ── 9. Final threshold tuning on best checkpoint ─────────────────────────────
print('Tuning decision threshold on best checkpoint...\n')
dev_probs, dev_true = get_probs(dev_loader)
best_f1, best_t = best_tuned_f1(dev_probs, dev_true)

print(f'Best threshold: {best_t:.2f}  |  Dev F1: {best_f1:.4f}\n')
dev_preds = (dev_probs >= best_t).astype(int)
print(classification_report(dev_true, dev_preds, target_names=['No-PCL', 'PCL'], digits=4))

In [ ]:
# ── 10. Write dev.txt ────────────────────────────────────────────────────────
with open('dev.txt', 'w') as f:
    for pred in dev_preds:
        f.write(f'{pred}\n')
print(f'dev.txt written  ({len(dev_preds)} lines)')

In [ ]:
# ── 11. Test set predictions → test.txt ──────────────────────────────────────
test_records  = load_test_tsv(os.path.join(DATA_DIR, 'task4_test.tsv'))
test_texts    = [t for _, t in test_records]
print(f'Test set: {len(test_texts)} samples')

test_dataset  = PCLDataset(test_texts)
test_loader   = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Reuse get_probs but it expects 'labels' key — make a labels-free version
model.eval()
test_probs = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        probs = torch.softmax(model(**batch).logits, dim=-1)[:, 1].cpu().numpy()
        test_probs.extend(probs)

test_preds = (np.array(test_probs) >= best_t).astype(int)
print(f'Predicted PCL: {test_preds.sum()} / {len(test_preds)} ({test_preds.mean()*100:.1f}%)')

with open('test.txt', 'w') as f:
    for pred in test_preds:
        f.write(f'{pred}\n')
print(f'test.txt written  ({len(test_preds)} lines)')

In [ ]:
# ── 12. Download output files ────────────────────────────────────────────────
from google.colab import files
files.download('dev.txt')
files.download('test.txt')
print('Done.')